# Used Functions

## Classification Model

In [ ]:
import torch
from tensorboard.summary import writer
from torch import nn

class IndependentHeadsMLP(nn.Module):
    """MLP with a shared body and independent per-label output heads."""

    def __init__(self, input_dim, hidden_dim, n_hidden, n_labels, activation="ReLU"):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList(
            [nn.Linear(hidden_dim, hidden_dim) for _ in range(n_hidden)]
        )
        self.heads = nn.ModuleList(
            [nn.Linear(hidden_dim, 1) for _ in range(n_labels)]
        )

        self.activation = nn.ReLU() if activation == "ReLU" else nn.Tanh()

    def forward(self, x):
        x = self.activation(self.input_layer(x))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        return torch.cat([head(x) for head in self.heads], dim=1)  # raw logits

## Training Functions
Along with TensorBoard logging

In [ ]:
from torch.utils.tensorboard import SummaryWriter


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
writer = SummaryWriter('runs/training1')

def _train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(loader)
    return avg_loss

def _calculate_val_loss(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for X,y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            output = model(X)
            loss = criterion(output, y)
            running_loss += loss.item()
    avg_loss = running_loss / len(loader)

    return avg_loss

def train_one_epoch(model, loader, optimizer, criterion, epoch:int):
    train_loss = _train_one_epoch(model, loader, optimizer, criterion)
    val_loss = _calculate_val_loss(model, loader, criterion)
    writer.add_scalars( main_tag='loss',
    tag_scalar_dict={'training': train_loss, 'validation': val_loss}, global_step=epoch)
    writer.flush()



# Model Training

## Getting Data Set

In [2]:
# Using MNIST DataSet
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np

mnist = fetch_openml('mnist_784', version=1, as_frame=False) # X: (70000, 784)
X, y = mnist["data"], mnist["target"]
X = X.astype(np.float32) / 255.0
y = y.astype(np.int64)
# scale to [0, 1]
X_trainval, X_test, y_trainval, y_test = train_test_split( X, y, test_size=5000, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=15000, random_state=42, stratify=y_trainval)

print(f"train size: {len(X_train)}")
print(f"val size: {len(X_val)}")
print(f"test size: {len(X_test)}")

train size: 50000
val size: 15000
test size: 5000


## Model Training
Data Preparation, Loaders and training loop